# Lab 09 — Blind Signal Separation, Built from Scratch

**Class 09.** Guide: `practice/09_blind_signal_separation.md`. Lecture: `slides/09_blind_signal_separation.pdf`.
Demos: `01_mixtures_pca.ipynb` to `05_audio.ipynb`. In class (60 minutes): Parts A to D; at home: Parts E and F.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready:
it draws the plots and prints the comparisons once your functions work.

**The rule of this lab.** Every separation method is a matrix factorisation $X \approx WH$ with a different
constraint. You write PCA, ICA and NMF with NumPy (the eigensolver and pyNNMF do the inner loop) and compare each
with its twin on the **same data**. The notebook prints the difference; explain it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from lab09_utils import (
    digits_matrix,
    kurtosis,
    masked_rmse,
    match,
    mixture_cloud,
    plot_mixtures,
    scatter_cloud,
    snr_db,
    split_ratings,
    toy_matrix,
    whiten,
)
from sklearn.decomposition import PCA, FastICA

warnings_hidden = None  # noqa: F841  (the notebook filters below)
import warnings  # noqa: E402

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
BLUE, ORANGE, RED = "#0072B2", "#E69F00", "#D55E00"

# Part A — PCA by Hand (12 min)

PCA is the eigendecomposition of the covariance of the **centred** data. For the two-microphone cloud the
eigenvector of the largest eigenvalue is the direction of most spread.

## A1. Centre, covariance, eigen

Implement `pca_by_hand(X)` (`X` is `(n, d)`; return the scores `T`, the components `V` (rows, unit length, ordered
by decreasing eigenvalue) and the eigenvalues `vals`). Centre inside; use `np.linalg.eigh` on the covariance.

In [ ]:
def pca_by_hand(X):
    Xc = X - X.mean(axis=0, keepdims=True)
    C = np.cov(Xc, rowvar=False)
    vals, vec = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    V = vec[:, order].T
    return Xc @ V.T, V, vals[order]

The driver compares your components, scores and explained variance with `PCA` on the mixture cloud.

In [ ]:
X, S, A = mixture_cloud()
Xn = X.T  # (n, 2): one row per sample, scikit-learn's convention
T, V, vals = pca_by_hand(Xn)
pca = PCA().fit(Xn)
rows = [
    ("eigenvalues", f"{vals[0]:.3f}, {vals[1]:.3f}"),
    ("PC1 direction", f"({V[0, 0]:.3f}, {V[0, 1]:.3f})"),
    ("twin PC1", f"({pca.components_[0, 0]:.3f}, {pca.components_[0, 1]:.3f})"),
    ("EVR", f"{vals[0] / vals.sum():.3f} / {vals[1] / vals.sum():.3f}"),
    ("twin EVR", f"{pca.explained_variance_ratio_[0]:.3f} / {pca.explained_variance_ratio_[1]:.3f}"),
]
print(pl.DataFrame(rows, schema=["quantity", "value"], orient="row"))
fig = scatter_cloud(X.T, w=V[0], title="the mixture cloud and PC1")
plt.show()

**Questions A1.**

1. Why must the data be centred before the covariance? What happens to PC1 if you forget it?
2. The two eigenvalues are very different. What does that say about how the two microphones are related?
3. PC1 points at the mixing direction, not at either source. Write in one sentence why, and predict what the
   second component adds.

**How we got there.** The covariance is $C = X_c^\top X_c / (n-1)$; `eigh` returns eigenvalues in increasing
order, so we reverse. The scores are the projections $T = X_c V$.

**Answers A1.** Without centring the mean dominates the outer product and PC1 chases the mean, not the spread.
The eigenvalue ratio is about 0.92/0.08: the two microphones are strongly correlated (both hear the same two
signals), so one direction explains almost everything. PC1 is the diagonal of the cloud because the mixing
matrix mixes both sources into both microphones; the second component (orthogonal to PC1) keeps only the small
leftover spread, which is why PCA separation of the sources is poor (about 0.3 dB in the lecture).

## A2. Reconstruction with $c$ components

Implement `reconstruct(T, V, c)` (keep the first $c$ components; return the reconstruction in the original
space, adding the mean back if you centred with it). The driver measures the relative error on the cloud and
on 64-pixel digits for $c = 1, 2, 8, 32$.

In [ ]:
def reconstruct(T, V, c):
    return T[:, :c] @ V[:c]

The driver prints the relative error $\|X - X_r\| / \|X\|$; for the cloud (centred inside `pca_by_hand`, so
also compare with the uncentred error) and for the digit images.

In [ ]:
Xd, yd = digits_matrix(400, 0)
print(
    f"cloud, c=2 relative error: {np.linalg.norm(Xn - reconstruct(T, V, 2)) / np.linalg.norm(Xn):.3f} (0 with both components)"
)
Tp, Vp, valsp = pca_by_hand(Xd)
rows = []
for c in (1, 2, 8, 32):
    err = np.linalg.norm(Xd - reconstruct(Tp, Vp, c)) / np.linalg.norm(Xd)
    rows.append((c, f"{err:.3f}", f"{sum(valsp[:c]) / valsp.sum():.3f}"))
print(pl.DataFrame(rows, schema=["c", "relative error", "explained variance"], orient="row"))
fig, axes = plt.subplots(1, 4, figsize=(7.0, 1.6))
for ax, c in zip(axes, (4, 8, 16, 32), strict=True):
    ax.imshow(reconstruct(Tp, Vp, c)[0].reshape(8, 8), cmap="grey")
    ax.set_title(f"c = {c}", fontsize=8)
    ax.axis("off")
plt.show()

**Questions A2.**

1. On digits, how much variance does c = 32 keep, and what does the reconstruction look like at c = 4?
2. The error keeps falling with c. Why is c = 32 still not the original image?
3. PCA reconstruction of a digit is a blur. Does the blur come from the eigensolver or from the constraint?

**How we got there.** Keeping $c$ components and projecting back is $X_r = T_{:, :c} V_{:c}$: a rank-$c$
approximation in the least-squares sense (the Eckart–Young theorem, the deck's "variance is reconstruction
error" frame).

**Answers A1/A2.** See the printed table: c = 32 keeps about 0.9 of the variance and c = 4 already reads as a
digit. c = 32 is not exact because 64 pixels need all 64 components; the last ones have tiny eigenvalues but
are not zero. The blur is the orthogonal-direction constraint: a PCA component is a whole-image average
(positive and negative strokes cancel), not a part.

# Part B — ICA by Hand (15 min)

ICA whitens, then finds the rotation that makes the two components **least Gaussian** — here, one kurtosis
gradient ascent per source direction.

## B1. Whiten and one kurtosis ascent

Implement `kurtosis_gradient(X, w, step=0.05, iters=200)` (`X` is `(2, n)` **rows are signals**; return the
final unit vector `w` after `iters` steps of gradient ascent on $|kurt(w^\top X)|$; renormalise `w` each step).
The kurtosis of the projected signal $z = w^\top X$ is $\frac{\mathbb{E}z^4}{(\mathbb{E}z^2)^2} - 3$;
after whitening $\mathbb{E}z^2 = 1$, so kurtosis is $\mathbb{E}z^4 - 3$.

In [ ]:
def kurtosis_gradient(X, w, step=0.05, iters=200):
    w = w / np.linalg.norm(w)
    for _ in range(iters):
        z = w @ X
        g = 4 * ((z**3) @ X.T) / X.shape[1]  # d E z^4 / dw; whitening gives E z^2 = 1, so kurt = E z^4 - 3
        w = w + step * g * np.sign(kurtosis(z.reshape(1, -1))[0])
        w = w / np.linalg.norm(w)
    return w

The driver whitens the mixtures, runs your ascent from three starts and compares the found directions with
`FastICA` and with the true unmixing $A^{-1}$. The score is the correlation with the true sources after
`match` (the permutation/sign fix every blind method needs).

In [ ]:
Zw = whiten(X)
starts = np.array([[1.0, 0.0], [0.0, 1.0], [0.7, 0.7]])
dirs = []
for w0 in starts:
    w = kurtosis_gradient(Zw, w0)
    dirs.append(w)
W_hand = np.array(dirs[:2])
S_hand = W_hand @ Zw
S_hand, corr_hand = match(S, S_hand)
ica = FastICA(n_components=2, random_state=0).fit_transform(X.T).T
S_ica, corr_ica = match(S, ica)
S_pca, corr_pca = match(S, PCA(2).fit_transform(X.T).T)
print(
    f"hand ascent kurtosis: {[f'{kurtosis((d @ Zw).reshape(1, -1))[0]:.2f}' for d in dirs]} (sources: {[f'{k:.2f}' for k in kurtosis(S)]})"
)
print(
    f"correlation with truth: hand {[f'{c:.3f}' for c in corr_hand]}, FastICA {[f'{c:.3f}' for c in corr_ica]}, PCA {[f'{c:.3f}' for c in corr_pca]}"
)
print(f"hand SNR: {[f'{s:.1f}' for s in snr_db(S, S_hand)]} dB, FastICA: {[f'{s:.1f}' for s in snr_db(S, S_ica)]} dB")
fig = plot_mixtures(X, S, S_hand)
plt.show()

**Questions B1.**

1. Why does the ascent start from a whitened `X`? What happens to kurtosis maximisation if the components
   still have different scales?
2. Your two found directions are not exactly orthogonal. Should they be? What makes FastICA's pair orthogonal?
3. The recovered sources match the truth up to sign and scale. Where in your code did you not need to care?

**How we got there.** Whitening makes the covariance the identity, so every remaining freedom is a rotation
and kurtosis is scale-free per unit direction; the ascent step renormalises `w` so the objective is over the
unit circle. `match` fixes the permutation and the sign — the two ambiguities of any blind method.

**Answers B1.** Without whitening, a long direction could inflate $\mathbb{E}z^2$ and fake non-Gaussianity.
The two sources need not be orthogonal directions of the whitened cloud, but deflation (one direction at a
time, projected out) or the symmetric orthogonalisation inside FastICA forces the mixing matrix to be
invertible; our two greedy ascents can, in principle, converge to the same direction (here the three starts
separate). Sign and scale never entered: correlation and SNR are invariant to both, which is exactly the
ambiguity the lecture showed.

# Part C — NMF Multiplicative Updates by Hand (15 min)

NMF minimises $\|X - WH\|_F^2$ with $W, H \geq 0$ by the updates
$W \leftarrow W \odot (XH^\top)/(WHH^\top)$ and $H \leftarrow H \odot (W^\top X)/(W^\top WH)$.

## C1. The update loop

Implement `nmf_by_hand(X, k, iters=200, seed=0)` (`X` is `(n, d)` non-negative; return `W` `(n, k)` and `H`
`(k, d)` after `iters` multiplicative updates from a positive random init; track the Frobenius cost each 20
iterations). Init with `np.random.default_rng(seed)` uniform(0.1, 1) of the right shapes.

In [ ]:
def nmf_by_hand(X, k, iters=200, seed=0):
    rng = np.random.default_rng(seed)
    W = rng.uniform(0.1, 1.0, (X.shape[0], k))
    H = rng.uniform(0.1, 1.0, (k, X.shape[1]))
    costs = []
    for it in range(iters + 1):
        costs.append((it, np.linalg.norm(X - W @ H) ** 2))
        H *= (W.T @ X) / (W.T @ W @ H)
        W *= (X @ H.T) / (W @ H @ H.T)
    return W, H, costs

The driver runs your updates on the toy matrix, checks the cost never rises, compares with `pynnmf.nmf_mu`
and shows the learned parts.

In [ ]:
T = toy_matrix()
W, H, costs = nmf_by_hand(T, 2)
rises = [f"{c1[1] - c0[1]:.3e}" for c0, c1 in zip(costs[:-1], costs[1:], strict=False) if c1[1] > c0[1] + 1e-9]
print(f"cost {costs[0][1]:.2f} -> {costs[-1][1]:.2f}; iterations where the cost rose: {len(rises)}")
T_hat, Wt, Ht, _ = pynnmf.nmf_mu(T, 2, n=200, seed=0, init="nndsvd")
print(f"pynnmf.nmf_mu cost: {np.linalg.norm(T - T_hat) ** 2:.2f} (different init and scaling of the objective)")
print(
    f"parts (columns of W): {W.min(axis=0)} >= 0, reconstruction error {np.linalg.norm(T - W @ H) / np.linalg.norm(T):.3f}"
)
fig, ax = plt.subplots(figsize=(4.0, 1.8))
ax.plot([c for _, c in costs], color=RED, lw=1.2)
ax.set_xlabel("iteration", fontsize=7)
ax.set_ylabel("Frobenius cost", fontsize=7)
ax.tick_params(labelsize=6)
plt.show()

**Questions C1.**

1. Why can the multiplicative update never make an entry negative? What is the cost of that guarantee?
2. Your cost and pyNNMF's cost differ at the same k. Which two things can differ between two minima?
3. Increase k to 4 on the toy (which has true rank 2). What happens to the error, and why is that not free?

**How we got there.** Both sides of each update are entrywise non-negative (X, W, H >= 0), so the ratio and
the product keep the sign; the update is gradient descent with the step chosen so negative entries cannot
appear (the derivation of the deck).

**Answers C1.** A product of non-negative matrices stays non-negative; the price is a fixed, conservative
step size — the cost can stall (local minima) where plain gradient descent would pass. Two minima can differ
in the permutation of the parts and the scale of W vs H (only the product WH is fixed). At k = 4 the error
drops but the parts split the two true ones into arbitrary pieces; the extra rank fits the noise term we
added, which is why k is chosen by validation, not by training error (Part E).

# Part D — Collaborative Filtering with Missing Values (12 min)

Real rating matrices have holes. Weighted NMF counts only the observed entries: minimise
$\|M \odot (R - WH)\|_F^2$ where M is 1 on observed cells. pyNNMF's `rwnmf` does exactly this with NaN
entries ignored.

## D1. Fill the holes and recommend

Implement `fill(R_train, k, seed=0)` (R_train has NaN on unknown cells; return the reconstruction $WH$ from
`pynnmf.rwnmf` with rank k) and `recommend(R_pred, R_train, user, k_top=3)` (return the indices of the k_top largest
predicted ratings of `user` among the cells that are NaN in `R_train`).

In [ ]:
def fill(R_train, k, seed=0):
    R_hat, _, _, _ = pynnmf.rwnmf(R_train, k, seed=seed, alpha=0.1)
    return R_hat


def recommend(R_pred, R_train, user, k_top=3):
    scores = np.where(np.isnan(R_train[user]), R_pred[user], -np.inf)
    return np.argsort(scores)[::-1][:k_top]

The driver splits the 6 x 8 ratings (70 % train), fills with your rank-1 model and reports the held-out
RMSE, the training RMSE and the top recommendations for user 0. The rank sweep repeats the lecture's finding:
with 28 training cells, low rank **is** the regulariser.

In [ ]:
R_train, R_test, M_train, M_test = split_ratings(frac_test=0.3, seed=0)
R_pred = fill(R_train, 1)
print(
    f"held-out RMSE: {masked_rmse(R_pred, R_test, M_test):.3f}, train RMSE: {masked_rmse(R_pred, np.where(M_train, R_train, 0), M_train):.3f}"
)
rows = []
for k in (1, 2, 3, 4):
    P = fill(R_train, k)
    rows.append(
        (k, f"{masked_rmse(P, R_test, M_test):.3f}", f"{masked_rmse(P, np.where(M_train, R_train, 0), M_train):.3f}")
    )
print(pl.DataFrame(rows, schema=["rank", "test RMSE", "train RMSE"], orient="row"))
top = recommend(R_pred, R_train, 0, 3)
print(f"user 0 unrated, top predictions: {top} -> {R_pred[0, top]}")
print(pl.DataFrame(np.round(R_pred, 2)))

**Questions D1.**

1. The test RMSE at rank 1 is far better than at rank 2. Why does the richer model lose on 8 test cells?
2. A user with no ratings at all still gets predictions. From where does the model take their taste?
3. What would overfitting look like here (describe it with the train/test RMSE pair, no run needed)?

**How we got there.** `rwnmf` weights the Frobenius cost by the mask: missing cells contribute nothing to
the gradient, and the factors W (users x tastes) and H (tastes x items) interpolate the holes from the
observed pattern.

**Answers D1.** With 28 observed cells a rank-2 model has 2(6+8)-4 free parameters competing with 8 test
numbers: it interpolates noise, train RMSE falls while test RMSE rises. An all-NaN user row still gets
predictions because H (the item tastes) is learned from the other users; the model predicts from the taste
groups, not from the row itself. Overfitting = train RMSE well below test RMSE, growing with rank — exactly
the sweep's 0.21 vs 0.71 at rank 1 against 0.05 vs 0.75 at rank 2.

# Part E — LSI, at Home

Build the term-document matrix of the demo 04 toy, run tf-idf + `TruncatedSVD` (rank 2) and find the two
topics. Then project a query ("goal keeper election") and rank the documents by cosine in the latent space.
The demo `04_lsi_cf.ipynb` section E2 is the model to follow.

# Part F — Challenge: the Use-Case Tournament

The same task goes to PCA, ICA and NMF with one score. `tournament(task)` returns the score of each
method for `"sound"` (spectra of three notes, higher is better), `"strokes"` (sparseness of digit parts,
higher), `"blink"` (error after removing an eye blink, lower) and `"impute"` (error on 25 % hidden pixels,
lower). Run the four, fill the table (task, winner, the constraint that explains it) and say which task
has no NMF score, and why. The demo `06_usecases.ipynb` draws each task.

In [ ]:
from lab09_utils import tournament

for task in ("sound", "strokes", "blink", "impute"):
    print(task, {m: round(v, 3) for m, v in tournament(task).items()})

**Answers F.** *sound*: NMF recovers the spectra (0.995), ICA 0.949, PCA 0.782 — the spectra are
non-negative, only NMF's are. *strokes*: NMF parts are sparse (0.608 against 0.377): additive pieces,
where PCA and ICA components are signed blurs. *blink*: ICA removes the blink almost exactly (RMSE 0.060),
PCA mixes it with the rhythms (0.535): the blink is an independent source, not the direction of most
variance; NMF has no score because the channels are signed. *impute*: iterative PCA 0.205 and NMF 0.212 are
close on dense pixels; NMF pulls ahead when most entries are missing (demo 06: ratings at 10 % observed, HALS 0.377 against 0.497).
PCA and ICA tie wherever only the subspace counts (compression, imputation): ICA rotates the PCA basis.

**Answers to the home parts are in `solutions/lab09_bss_solution.ipynb`.**